In [8]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Intro2DF") \
    .getOrCreate()

In [9]:
spark

In [10]:
df = spark.read.csv("./data/large_dataset.csv", header=True, inferSchema=True)
df = df.sample(withReplacement=False, fraction=0.01, seed=42)
df.describe().show()

+-------+--------------------+----------------+--------+----+-----------------+--------------------+-------------------+--------------------+-----------+-------+-----------+------------------+----------+-----------------+
|summary|                  id|            name|username| sex|              age|               email|       phone_number|             address|       city|  state|    country|           company|occupation|           salary|
+-------+--------------------+----------------+--------+----+-----------------+--------------------+-------------------+--------------------+-----------+-------+-----------+------------------+----------+-----------------+
|  count|                 556|             556|     556| 556|              556|                 556|                556|                 556|        556|    556|        556|               556|       556|              556|
|   mean|                NULL|            NULL|    NULL|NULL|59.17985611510792|                NULL|5.7739295934

In [11]:
# df = spark.read.csv("./data/large_dataset.csv", header=True, inferSchema=True)
# df.show()

In [12]:
# Describe the DataFrame

df.describe().show()

+-------+--------------------+----------------+--------+----+-----------------+--------------------+-------------------+--------------------+-----------+-------+-----------+------------------+----------+-----------------+
|summary|                  id|            name|username| sex|              age|               email|       phone_number|             address|       city|  state|    country|           company|occupation|           salary|
+-------+--------------------+----------------+--------+----+-----------------+--------------------+-------------------+--------------------+-----------+-------+-----------+------------------+----------+-----------------+
|  count|                 556|             556|     556| 556|              556|                 556|                556|                 556|        556|    556|        556|               556|       556|              556|
|   mean|                NULL|            NULL|    NULL|NULL|59.17985611510792|                NULL|5.7739295934

In [13]:
# Selecting specific columns

df.select("name", "age").show()

+------------------+---+
|              name|age|
+------------------+---+
|     Amber Johnson|  6|
|       Lynn Guzman| 75|
|       Dana Miller| 89|
|    Laura Williams| 59|
|    Julia Williams| 17|
|    Caitlin Weaver| 50|
|       David Moore| 70|
|       John Wright| 32|
|        Karen Bray| 37|
|  Victoria Wilkins| 14|
|      James Steele| 28|
|      Eric Rowland| 22|
|     Megan Collins| 20|
| Alexandra Beasley| 42|
|   Roberto Gregory| 43|
|Christina Bartlett| 90|
|       Gregory Fox| 34|
|    Philip Andrade|108|
|     Joshua Miller| 30|
|      Sheri Dodson| 61|
+------------------+---+
only showing top 20 rows


In [14]:
from pyspark.sql.functions import col

# Filtering rows based on a condition
df.filter(col("age") > 30).show()

+--------------------+------------------+---------------+---+----------+---+--------------------+--------------------+--------------------+------------------+--------------+--------------------+--------------------+----------+------+
|                  id|              name|       username|sex|       dob|age|               email|        phone_number|             address|              city|         state|             country|             company|occupation|salary|
+--------------------+------------------+---------------+---+----------+---+--------------------+--------------------+--------------------+------------------+--------------+--------------------+--------------------+----------+------+
|52d8e4c8-dfbe-486...|       Lynn Guzman|     allenscott|  F|1951-07-04| 75|rogersnicholas@ex...|       (251)907-7512|Unit 0716 Box 445...|         Aprilbury| West Virginia|         Afghanistan|       Mitchell-King|   Teacher| 68264|
|ae2d64cf-dbab-4bc...|       Dana Miller|         gary96|  M|193

In [15]:
# Adding a new column with a calculation

df.withColumn("salary_plus_tax", col("salary") * 1.1).show()

+--------------------+------------------+---------------+---+----------+---+--------------------+--------------------+--------------------+------------------+-------------+--------------------+--------------------+----------+------+------------------+
|                  id|              name|       username|sex|       dob|age|               email|        phone_number|             address|              city|        state|             country|             company|occupation|salary|   salary_plus_tax|
+--------------------+------------------+---------------+---+----------+---+--------------------+--------------------+--------------------+------------------+-------------+--------------------+--------------------+----------+------+------------------+
|972f637b-239d-447...|     Amber Johnson|         fyoung|  M|2020-12-01|  6| mhoward@example.com|   (771)454-5885x135|947 Gonzales Park...|         Smithfurt|         Utah|             Jamaica|         Meyer-Hardy|    Doctor|148309|163139.90000

In [16]:
from pyspark.sql.functions import avg

# Grouping by a column and calculating an aggregate
df.groupBy("occupation").agg(avg("salary")).show()

+----------+-----------------+
|occupation|      avg(salary)|
+----------+-----------------+
| Scientist|95756.85606060606|
|   Teacher|  84942.793814433|
|    Artist| 87884.4695652174|
|    Doctor|91625.95327102803|
|  Engineer|          92522.0|
+----------+-----------------+



In [17]:
# Counting the number of occurrences of each city and ordering by count

df.groupBy("sex").count().show()

+---+-----+
|sex|count|
+---+-----+
|  F|  260|
|  M|  296|
+---+-----+



In [18]:
# Counting the number of occurrences of each city and ordering by count

df.groupBy("city").count().orderBy(col("count").desc()).show(5)

+------------+-----+
|        city|count|
+------------+-----+
|  East James|    2|
|  Port Jason|    2|
| Greeneshire|    2|
|  New Joseph|    2|
|Rebeccamouth|    2|
+------------+-----+
only showing top 5 rows


In [19]:
# Calculating average age by occupation

df.groupBy("occupation").agg(avg("age").alias("avg_age")).show()

+----------+------------------+
|occupation|           avg_age|
+----------+------------------+
| Scientist| 56.90151515151515|
|   Teacher| 55.08247422680412|
|    Artist| 62.21739130434783|
|    Doctor| 60.35514018691589|
|  Engineer|61.304761904761904|
+----------+------------------+



In [21]:
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType
import os
import sys

# Ép PySpark sử dụng đúng file python.exe của môi trường hiện tại
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

from pyspark.sql import SparkSession
# Khởi tạo SparkSession của bạn ở dưới đây...

# Defining a UDF to categorize age
def categorize_age(age):
    if age < 20:
        return "teenager"
    elif 20 <= age < 60:
        return "adult"
    else:
        return "senior"
    
categorize_age_udf = udf(categorize_age, StringType())

# Applying the UDF to create a new column
df.withColumn("age_category", categorize_age_udf(col("age"))) \
    .select("age", "age_category") \
    .coalesce(6) \
    .show()

+---+------------+
|age|age_category|
+---+------------+
|  6|    teenager|
| 75|      senior|
| 89|      senior|
| 59|       adult|
| 17|    teenager|
| 50|       adult|
| 70|      senior|
| 32|       adult|
| 37|       adult|
| 14|    teenager|
| 28|       adult|
| 22|       adult|
| 20|       adult|
| 42|       adult|
| 43|       adult|
| 90|      senior|
| 34|       adult|
|108|      senior|
| 30|       adult|
| 61|      senior|
+---+------------+
only showing top 20 rows


In [22]:
df.explain()

== Physical Plan ==
*(1) Sample 0.0, 0.01, false, 42
+- FileScan csv [id#17,name#18,username#19,sex#20,dob#21,age#22,email#23,phone_number#24,address#25,city#26,state#27,country#28,company#29,occupation#30,salary#31] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/c:/Project/Some_Ai_Stuff/data_engineer/PySpark_examples/data/lar..., PartitionFilters: [], PushedFilters: [], ReadSchema: struct<id:string,name:string,username:string,sex:string,dob:date,age:int,email:string,phone_numbe...




In [23]:
df.rdd.getNumPartitions()

3

In [24]:
df = df.repartition(24)

df.rdd.getNumPartitions()

24

In [25]:
df.cache().show()

+--------------------+-------------------+----------------+---+----------+---+--------------------+--------------------+--------------------+-------------------+-------------+-----------------+--------------------+----------+------+
|                  id|               name|        username|sex|       dob|age|               email|        phone_number|             address|               city|        state|          country|             company|occupation|salary|
+--------------------+-------------------+----------------+---+----------+---+--------------------+--------------------+--------------------+-------------------+-------------+-----------------+--------------------+----------+------+
|b76525db-49d7-47f...|        Kyle Chavez|      seanhoward|  F|1972-01-24| 54|toddchristopher@e...|        698.373.1160|23473 Charles Sho...|         Lake Shawn|     Nebraska|        Venezuela|        Montes-Smith|    Artist| 67971|
|520c3e18-fd88-48b...|       Michael Neal|   riveragregory|  M|1932-

In [26]:
from pyspark import StorageLevel

df.persist(StorageLevel.DISK_ONLY)

DataFrame[id: string, name: string, username: string, sex: string, dob: date, age: int, email: string, phone_number: string, address: string, city: string, state: string, country: string, company: string, occupation: string, salary: int]

In [27]:
df.show()

+--------------------+-------------------+----------------+---+----------+---+--------------------+--------------------+--------------------+-------------------+-------------+-----------------+--------------------+----------+------+
|                  id|               name|        username|sex|       dob|age|               email|        phone_number|             address|               city|        state|          country|             company|occupation|salary|
+--------------------+-------------------+----------------+---+----------+---+--------------------+--------------------+--------------------+-------------------+-------------+-----------------+--------------------+----------+------+
|b76525db-49d7-47f...|        Kyle Chavez|      seanhoward|  F|1972-01-24| 54|toddchristopher@e...|        698.373.1160|23473 Charles Sho...|         Lake Shawn|     Nebraska|        Venezuela|        Montes-Smith|    Artist| 67971|
|520c3e18-fd88-48b...|       Michael Neal|   riveragregory|  M|1932-

In [ ]:
# spark.sparkContext.setCheckpointDir("./checkpoints")

# df = df.checkpoint()